# TARANG: Grating Couplers

| Section | Functions | Credits |
|---|---|---|
| Geometry | `GratingConfig`, `grating_coupler.get_active_simulation`, `display_2d`, `display_3d` | none |
| Efficiency, spectrum and fields | `GratingCoupler.get_efficiency`, `get_spectrum_summary`, `get_fields` | yes |
| Phase matching | `phase_matched_period` | none |
| Bayesian design search | `nested_sampling_example.py`, `post_process_run`, `plot_corner` | capped |

Start in 2D (`is_3d=False`): it is much cheaper than 3D and good for choosing period, fill
factor and etch depth.

**Fibre orientation.** The waveguide leaves the grating towards -x. With the default
`fiber_tilt="towards"` the beam travels towards the waveguide (the fibre's upper end leans away
from it; two fibres feeding mirrored couplers form a V), and the first-order period is
λ / (n_eff − sin θ). `fiber_tilt="away"` gives λ / (n_eff + sin θ); it was the only
orientation before v0.1.0. `source_x` is the beam centre on the grating surface.

In [ ]:
# ---- Logging ---- #
from tarang import configure_logging, set_tidy3d_output

configure_logging(level="INFO")
set_tidy3d_output(False)

# ---- Importing Media ---- #
# The coupler below works at 1.55 µm: fit the materials over that band.
from tarang.media import get_medium

SiN = get_medium(
    "si3n4",
    (1.350, 1.750),
)

SiO2 = get_medium(
    "sio2",
    (1.350, 1.750)
)

Si = get_medium(
    "si_franta",     # Si data covering 1550 nm (see media/data/SOURCES.md)
    (1.350, 1.750),
    tolerance = 1e-5
)

Air = get_medium(
    "air",
    (1.350, 1.750)
)

In [ ]:
# ---- Cost cap for this notebook ---- #
# Every cloud run is priced before it starts. A run whose estimate is above this
# limit (FlexCredits) is deleted unstarted and raises CostLimitExceededError, so
# nothing is charged. Raise it deliberately when you need a bigger run.
import os
os.environ["TARANG_MAX_COST"] = "1.0"     # per run

## Grating Coupler: Example Geometry

In [ ]:
# ---- Defining the Geometry of the Grating Coupler ---- #

from tarang.geometries import grating_coupler
from tarang.utilities import sim_plotter
from tarang.grating_coupler import GratingConfig, GratingCoupler, phase_matched_period

from pathlib import Path
import numpy as np

example_config = GratingConfig(
    wavelength_um=1.550,    # Central wavelength (μm)
    theta_deg=12.0,         # Fibre angle from vertical (degrees)
    fiber_tilt="towards",   # Beam travels towards the waveguide (default)
    mfd=10.4,               # Mode Field Diameter (Standard SMF-28)
    source_x=4.5,           # Beam centre on the grating surface (μm from its start)
    source_gap=2.0,         # Distance between fiber source and top cladding

    # Substrate & Layer Geometry
    core_height=0.800,      # Core thickness (μm)
    top_cladding_height=2.0,    # Air/Cladding buffer height above core
    box_height=4.0,         # Buried Oxide (BOX) thickness
    handle_height=8.0,   # Silicon handle thickness

    # Grating Teeth Dimensions
    etch_depth=0.5136,      # Partial etch depth (μm)
    period=1.0516,          # Grating period (μm): λ/(n_eff − sin θ) with n_eff = 1.682
    ff=0.5136,              # Fill factor (50% tooth, 50% groove)
    n_teeth=22,             # Total number of grating teeth
    w_grating=10.4,        # Transverse width for 3D runs

    # Source & Domain Settings
    pol_angle=np.pi/2,          # 0.0 = TE polarization
    buffer=3.0              # Domain margin buffer
)

# Starting period from phase matching (n_eff = 1.682 is the average index of this partially
# etched 800 nm grating; the simulation finds the exact peak):
print(f"phase-matched period: {phase_matched_period(1.682, 1.550, 12.0, 'towards'):.4f} um")

example_grating_coupler_3d: GratingCoupler = grating_coupler.get_active_simulation(
    config=example_config,
    core_medium=SiN,
    box_medium=SiO2,
    handle_medium=None,  
    top_cladding_medium=Air,    # Air-clad (or pass SiO2 for oxide clad)
    is_3d=True           
)

example_grating_coupler_2d : GratingCoupler = grating_coupler.get_active_simulation(
    config=example_config,
    core_medium=SiN,
    box_medium=SiO2,
    handle_medium=None,
    top_cladding_medium=Air,
    is_3d=False
)

sim_plotter.display_3d(
    device = example_grating_coupler_3d,
    output_path = Path("example_results/grating_coupler/example_3d.png"),
    show = True,
    save = True
)

sim_plotter.display_2d(
    device=example_grating_coupler_2d,
    output_path=Path("example_results/grating_coupler/example_2d.png"),
    show = True,
    save = True
)

## Grating Coupler: Analytics

### Grating Coupler Efficiency

In [ ]:
# ---- Grating Coupler Analytics ---- #

# 1) Coupler efficiency
# The folder passed here keeps the results. They are stored under a fingerprint of
# the complete simulation (materials included): asking again for the same coupler
# loads them from disk; changing anything runs a new simulation. Each run is priced
# first and stops above the cost cap set at the top of this notebook.
example_grating_coupler_2d.get_efficiency(Path("example_results/grating_coupler/"))
example_grating_coupler_3d.get_efficiency(Path("example_results/grating_coupler/"))

# 2) Spectrum: the mode monitor records ±bandwidth_um (default ±100 nm) around the design
# wavelength. The summary gives the peak wavelength, the peak efficiency, the efficiency at
# 1550 nm and the 1 dB / 3 dB bandwidths (from the same 2D result: no extra run).
summary = example_grating_coupler_2d.get_spectrum_summary(Path("example_results/grating_coupler/"))
for key, value in summary.items():
    print(f"{key:24s} {value}")

### Field plots

In [ ]:
# 2.1) Visualize fields within the 2D coupler
example_grating_coupler_2d.get_fields(Path("example_results/grating_coupler/"))

In [ ]:
# 2.2) Visualize fields within the 3D coupler
example_grating_coupler_3d.get_fields(Path("example_results/grating_coupler/"))

## Bayesian design search: results

The cells below read the results of `nested_sampling_example.py` (run it first from this `examples` folder; it spends FlexCredits up to its budget cap). They only read files, and print a note instead of failing when no run exists yet.

In [ ]:
from pathlib import Path
from tarang.utilities.nested_sampling_post_process import post_process_run

run_dir = Path("example_results/bayesian_demo_budgeted")   # written by nested_sampling_example.py
if (run_dir / "chains").exists():
    results = post_process_run(output_dir=run_dir, show=True)
    display(results)
else:
    print(f"No nested-sampling run in {run_dir} yet: run nested_sampling_example.py first.")


### Corner plot from an ongoing or saved nested-sampling run
Run these cells independently: only the run directory is needed. While a nested-sampling search (dynesty or MultiNest) runs in another kernel, use this notebook in a separate kernel to inspect its periodically flushed files. `source="auto"` prefers a posterior chain; `source="live"` selects physical live points and labels them as a diagnostic snapshot. Early runs may not have enough flushed rows yet.

In [ ]:
from pathlib import Path
from tarang.utilities import plot_corner

run_dir = Path("example_results/bayesian_demo_budgeted")   # written by nested_sampling_example.py
if (run_dir / "chains").exists():
    corner_result = plot_corner(directory=run_dir / "chains", source="auto", show=True)
else:
    print(f"No chains in {run_dir} yet.")


In [ ]:
# Run this cell when you specifically want the current live-point distribution.
if (run_dir / "chains").exists():
    live_result = plot_corner(
        directory=run_dir / "chains",
        source="live",
        output_path=run_dir / "live_snapshot.png",
        show=True,
    )
else:
    print(f"No chains in {run_dir} yet.")
